# Policy Compliance Review with AutoGen: rules in code, judgement by agents

A multi-agent system that reviews company policy documents for compliance issues, decides who must act on each one, and **measures how accurate it is** against an answer key.

| Part | Who does it | Why |
|---|---|---|
| **Rule checks**: required sections, signatures, document age | **Code**, from a rules table per document type | These are yes/no facts. Code checks them the same way every time and can't miss one |
| **Judgement issues**: outdated references, weak security controls, contradictions | **Analyst agent**, using tools to read the full document and the rule results | These need reading and reasoning |
| **Quality control** | **Reviewer agent** (maker–checker), then **code** verifies every quoted piece of evidence | Removes duplicates, weak findings and anything not actually in the document |
| **Severity and routing**: who must act | **Code**, from fixed tables | A routing decision shouldn't depend on how an LLM feels that day |

Built with **AutoGen (AG2 Classic)**: tool calling with `register_function`, and a `GroupChat` whose speaking order is set by a **custom speaker-selection function**, so every review follows the same path.

**Setup:** add `OPENAI_API_KEY` to Colab **Secrets** (🔑) with notebook access on. (Azure OpenAI also works; see Step 5.) Steps 1–4 run without any key.

## Step 1: Install AutoGen (AG2 Classic)

This notebook uses the **classic AutoGen API** (`AssistantAgent`, `GroupChat`, `register_function`), imported as `autogen`. It is installed with the `autogen` package.

Don't install `ag2` for this notebook: since version 1.0, the `ag2` package is a different framework with new imports (`import ag2`) and no `autogen` module, which causes `ModuleNotFoundError: No module named 'autogen'`.

The second line upgrades `protobuf`. Colab comes with Google libraries built for a newer `protobuf` than the one in the runtime, and AutoGen imports them while checking which model providers are available. Without the upgrade, `import autogen` fails with *"Detected incompatible Protobuf Gencode/Runtime versions"*.

**After installing, use Runtime ▸ Restart session** (required: the old `protobuf` stays loaded until you restart), then continue from Step 2.

In [ ]:
!pip install -q -U "autogen[openai]" pandas
!pip install -q -U "protobuf>=6.33.5,<7"

## Step 2: The documents and the answer key

Five sample policies. Issues were **planted on purpose** and recorded in `ANSWER_KEY`, so the system's output can be scored. Two kinds of issue are planted:

* **Rule issues**: a missing required section, a missing signature, a document older than its review cycle.
* **Judgement issues**: references to retired documents, weak security controls, and statements that contradict each other. No simple rule can catch these.

`DOC004` is deliberately **compliant**: a good system should find nothing there.

Document age is measured from a fixed date (`AS_OF`), so the answer key doesn't change when you run the notebook later.

In [ ]:
from datetime import date

AS_OF = date(2026, 10, 21)

DOCUMENTS = {
    "DOC001": {
        "title": "Employee Remote Work Policy", "type": "HR Policy", "last_updated": "2025-01-15",
        "content": """Remote Work Policy
Effective Date: January 15, 2025
1. Eligibility: All full-time employees after 90 days of employment.
2. Equipment: The company provides a laptop. Request accessories with form HR-17 from the 2019 Employee Handbook, which was retired in 2022.
3. Work Hours: Standard hours are 9:00-17:00 EST, with core hours of 10:00-15:00.
4. Flexibility: Remote employees may set their own hours; there are no core hours.
Approved by: Dana Whitfield, VP People""",
    },
    "DOC002": {
        "title": "Data Security Manual", "type": "Technical Manual", "last_updated": "2026-02-10",
        "content": """Data Security Manual
Version: 3.0
Security Requirements:
- Passwords must be at least 8 characters and are rotated every 30 days.
- Sensitive data at rest is encrypted with AES-256.
- Access to production systems requires manager approval.""",
    },
    "DOC003": {
        "title": "Customer Service Guidelines", "type": "Operational Guidelines", "last_updated": "2026-06-01",
        "content": """Customer Service Guidelines
Procedures: All customer tickets must receive a first response within 24 hours.
Escalation: Unresolved tickets go to a supervisor after 48 hours.
Response Standards: Standard tickets receive a first response within 3 business days.
Communication: Use the template responses from the 2019 Support Handbook.
Approved by: [Missing Signature]
Review Date: [Not Specified]""",
    },
    "DOC004": {
        "title": "Expense Reimbursement Policy", "type": "HR Policy", "last_updated": "2026-08-01",
        "content": """Expense Reimbursement Policy
Effective Date: August 1, 2026
1. Eligibility: All employees who incur approved business expenses.
2. Submission: Claims are submitted in the expense system within 30 days, with itemised receipts.
3. Limits: Meals up to $75 per day; economy class for flights under 6 hours.
Approved by: Marcus Lee, Chief Financial Officer""",
    },
    "DOC005": {
        "title": "Vendor Access Standard", "type": "Technical Manual", "last_updated": "2026-09-15",
        "content": """Vendor Access Standard
Version: 1.2
Security Requirements:
- Vendor accounts require multi-factor authentication and expire after 90 days.
- To simplify onboarding, a vendor team may share one login across its staff.
Incident Response: Vendors must report security incidents to the security team within 24 hours.""",
    },
}

# Planted issues: (document, issue code). Rule codes come from the rules table; the rest need judgement.
ANSWER_KEY = {
    ("DOC001", "STALE_DOCUMENT"), ("DOC001", "OUTDATED_REFERENCE"), ("DOC001", "INCONSISTENCY"),
    ("DOC002", "STALE_DOCUMENT"), ("DOC002", "MISSING_SECTION:Incident Response"), ("DOC002", "WEAK_CONTROL"),
    ("DOC003", "MISSING_SIGNATURE"), ("DOC003", "MISSING_SECTION:Review Date"),
    ("DOC003", "OUTDATED_REFERENCE"), ("DOC003", "INCONSISTENCY"),
    ("DOC005", "WEAK_CONTROL"),
}
JUDGEMENT_TYPES = ["OUTDATED_REFERENCE", "WEAK_CONTROL", "INCONSISTENCY"]

print(f"{len(DOCUMENTS)} documents, {len(ANSWER_KEY)} planted issues "
      f"({sum(not c.startswith(tuple(JUDGEMENT_TYPES)) for _, c in ANSWER_KEY)} rule, "
      f"{sum(c in JUDGEMENT_TYPES for _, c in ANSWER_KEY)} judgement); DOC004 is compliant")

## Step 3: Rules, severity and routing, all in code

* `RULES`: what each document type must contain, whether it needs a signature, and how often it must be reviewed.
* `check_rules`: applies them. A section only counts if it has a real value, so `Review Date: [Not Specified]` is reported as missing.
* `SEVERITY` and `ROUTES`: map each issue type to a severity, and each document's most severe issue to who must act. The LLM never decides these.
* `verify_evidence`: every finding from the agents must quote the document word for word. Findings whose quote isn't in the document are rejected.

In [ ]:
import json
import re
from dataclasses import dataclass, asdict

RULES = {
    "HR Policy":              {"required_sections": ["Effective Date", "Eligibility"], "requires_signature": True, "max_age_months": 12},
    "Technical Manual":       {"required_sections": ["Version", "Security Requirements", "Incident Response"], "requires_signature": False, "max_age_months": 6},
    "Operational Guidelines": {"required_sections": ["Procedures", "Escalation", "Review Date"], "requires_signature": True, "max_age_months": 12},
}

SEVERITY = {"WEAK_CONTROL": "Critical", "MISSING_SIGNATURE": "Major", "MISSING_SECTION": "Major",
            "INCONSISTENCY": "Major", "STALE_DOCUMENT": "Minor", "OUTDATED_REFERENCE": "Minor", "OTHER": "Minor"}
SEVERITY_RANK = {"Critical": 3, "Major": 2, "Minor": 1}
ROUTES = {"Critical": "Escalate to Legal & Executive review",
          "Major": "Route to department head for approval",
          "Minor": "Send revision request to document owner",
          None: "No action: compliant"}

PLACEHOLDER = re.compile(r"^\[.*\]$|^(tbd|n/?a|none|not specified)$", re.IGNORECASE)


@dataclass
class Finding:
    doc_id: str
    code: str              # e.g. MISSING_SECTION:Review Date, WEAK_CONTROL
    source: str            # "rule" or "agent"
    evidence: str          # exact text from the document, or what is missing
    recommendation: str

    @property
    def issue_type(self) -> str:
        return self.code.split(":")[0]

    @property
    def severity(self) -> str:
        return SEVERITY.get(self.issue_type, "Minor")


def field_value(content: str, name: str):
    """The value after 'Name:' on its own line (optionally numbered), or None."""
    match = re.search(rf"^\s*(?:\d+\.\s*)?{re.escape(name)}\s*:\s*(.*)$", content, re.IGNORECASE | re.MULTILINE)
    return match.group(1).strip() if match else None


def months_between(earlier: date, later: date) -> int:
    return (later.year - earlier.year) * 12 + later.month - earlier.month - (later.day < earlier.day)


def check_rules(doc_id: str, as_of: date = AS_OF) -> list:
    doc, findings = DOCUMENTS[doc_id], []
    rules = RULES[doc["type"]]
    for section in rules["required_sections"]:
        value = field_value(doc["content"], section)
        if not value or PLACEHOLDER.match(value):
            findings.append(Finding(doc_id, f"MISSING_SECTION:{section}", "rule",
                                    f"'{section}' is {'a placeholder: ' + value if value else 'absent'}",
                                    f"Add a completed '{section}' section."))
    if rules["requires_signature"]:
        signer = field_value(doc["content"], "Approved by")
        if not signer or PLACEHOLDER.match(signer):
            findings.append(Finding(doc_id, "MISSING_SIGNATURE", "rule",
                                    f"Approval is {'a placeholder: ' + signer if signer else 'absent'}",
                                    "Obtain and record a named approver."))
    age = months_between(date.fromisoformat(doc["last_updated"]), as_of)
    if age > rules["max_age_months"]:
        findings.append(Finding(doc_id, "STALE_DOCUMENT", "rule",
                                f"Last updated {doc['last_updated']} ({age} months ago; review cycle is {rules['max_age_months']} months)",
                                "Review and re-approve the document."))
    return findings


def normalise(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower().replace("’", "'").replace("–", "-")).strip()


def verify_evidence(doc_id: str, quote: str) -> bool:
    """True if the quote appears word for word in the document (ignoring case and spacing)."""
    q = normalise(quote).strip(" .\"'")
    return len(q) >= 8 and q in normalise(DOCUMENTS[doc_id]["content"])


def route(findings: list):
    top = max((f.severity for f in findings), key=SEVERITY_RANK.get, default=None)
    return top, ROUTES[top]


for doc_id in DOCUMENTS:
    found = check_rules(doc_id)
    print(f"{doc_id}: {[f.code for f in found] or 'no rule issues'}")

## Step 4: Offline tests (no API key)

The rule engine, evidence check, routing, the agents' speaking order and the scoring are tested before any LLM is called. The speaking-order function is defined here too, so it can be tested with stand-in agents.

In [ ]:
from types import SimpleNamespace


def make_speaker_selector(starter, analyst, tool_executor, reviewer):
    """Fixed path: starter → Analyst ⇄ ToolExecutor (as often as the Analyst calls tools) → Reviewer → end."""
    def select(last_speaker, groupchat):
        last = groupchat.messages[-1] if groupchat.messages else {}
        if last_speaker is starter:
            return analyst
        if last_speaker is analyst:
            return tool_executor if last.get("tool_calls") else reviewer
        if last_speaker is tool_executor:
            return analyst
        return None          # after the Reviewer: nobody speaks; the manager also stops on TERMINATE
    return select


def extract_json(text: str):
    """The first {...} block in a reply, parsed; None if there isn't a valid one."""
    match = re.search(r"\{.*\}", text or "", re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def score(predicted: set, answer_key: set = ANSWER_KEY, kinds=None) -> dict:
    """Recall against the answer key, overall or for one kind ('rule' / 'judgement')."""
    def kind(code):
        return "judgement" if code in JUDGEMENT_TYPES else "rule"
    key = {k for k in answer_key if kinds is None or kind(k[1]) == kinds}
    pred = {p for p in predicted if kinds is None or kind(p[1]) == kinds}
    hits = key & pred
    return {"found": len(hits), "planted": len(key), "missed": sorted(key - pred),
            "not_in_key": sorted(pred - answer_key)}


# 1. Rule engine reproduces every planted rule issue, and nothing else
rule_found = {(f.doc_id, f.code) for d in DOCUMENTS for f in check_rules(d)}
assert rule_found == {k for k in ANSWER_KEY if k[1] not in JUDGEMENT_TYPES}, rule_found
assert check_rules("DOC004") == []
assert check_rules("DOC002", as_of=date(2026, 7, 1)) == [f for f in check_rules("DOC002") if f.code != "STALE_DOCUMENT"]
print("✓ 1. rules: all 5 planted rule issues found, none extra; age depends on AS_OF")

# 2. Evidence must be in the document
assert verify_evidence("DOC003", "use the template responses from the 2019 support handbook")
assert verify_evidence("DOC005", "a vendor team may share one login   across its staff")
assert not verify_evidence("DOC005", "vendors may share passwords freely")
assert not verify_evidence("DOC001", "2019")                       # too short to prove anything
print("✓ 2. evidence: real quotes accepted, invented or trivial quotes rejected")

# 3. Severity and routing
assert route(check_rules("DOC004")) == (None, "No action: compliant")
assert route([Finding("DOC005", "WEAK_CONTROL", "agent", "x", "y")])[0] == "Critical"
assert route(check_rules("DOC003"))[0] == "Major"
print("✓ 3. routing: most severe issue decides who must act")

# 4. Speaking order, with stand-in agents
starter, analyst, executor, reviewer = (SimpleNamespace(name=n) for n in ("Starter", "Analyst", "Executor", "Reviewer"))
select = make_speaker_selector(starter, analyst, executor, reviewer)
chat = SimpleNamespace(messages=[{"content": "review DOC001"}])
assert select(starter, chat) is analyst
chat.messages.append({"content": None, "tool_calls": [{"id": "1"}]}); assert select(analyst, chat) is executor
chat.messages.append({"content": "tool output"});                    assert select(executor, chat) is analyst
chat.messages.append({"content": '{"findings": []}'});              assert select(analyst, chat) is reviewer
assert select(reviewer, chat) is None
print("✓ 4. speaker order: Analyst ⇄ ToolExecutor until it stops calling tools, then Reviewer, then end")

# 5. JSON extraction and scoring
assert extract_json('Here you go: {"a": 1}\nTERMINATE') == {"a": 1} and extract_json("no json") is None
perfect = score(ANSWER_KEY)
assert perfect["found"] == len(ANSWER_KEY) and perfect["missed"] == [] and perfect["not_in_key"] == []
partial = score({("DOC001", "INCONSISTENCY"), ("DOC004", "WEAK_CONTROL")}, kinds="judgement")
assert partial["found"] == 1 and partial["planted"] == 6 and partial["not_in_key"] == [("DOC004", "WEAK_CONTROL")]
print("✓ 5. scoring: recall and findings outside the answer key computed correctly")
print("\nAll offline tests passed.")

## Step 5: Connect to the LLM

`PROVIDER = "openai"` uses `OPENAI_API_KEY`. For Azure OpenAI, set `PROVIDER = "azure"` and add `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_DEPLOYMENT` as Colab secrets, so your resource name stays out of the notebook.

No output-token limit is set: with reasoning models such as gpt-5-mini, a low limit can be used up by internal reasoning and leave the visible reply empty.

In [ ]:
import os


def load_secret(name: str):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name)


PROVIDER = "openai"          # or "azure"
MODEL = "gpt-4o-mini"        # for Azure, the deployment name is read from AZURE_OPENAI_DEPLOYMENT

if PROVIDER == "openai":
    config = {"model": MODEL, "api_key": load_secret("OPENAI_API_KEY"), "api_type": "openai"}
else:
    config = {"model": load_secret("AZURE_OPENAI_DEPLOYMENT"), "api_key": load_secret("AZURE_OPENAI_API_KEY"),
              "base_url": load_secret("AZURE_OPENAI_ENDPOINT"), "api_type": "azure", "api_version": "2024-10-21"}
missing = [k for k, v in config.items() if not v]
if missing:
    raise RuntimeError(f"Missing settings for {PROVIDER}: {missing}. Add them as Colab secrets.")

llm_config = {"config_list": [config], "cache_seed": None}   # no caching: every run calls the model
print(f"Using {PROVIDER} / {config['model']}")

## Step 6: The agents and their tools

| Agent | Type | Role |
|---|---|---|
| `Starter` | UserProxyAgent | Opens each review with the document ID |
| `Analyst` | AssistantAgent | Reads the full document and the rule results **through tools**, then drafts judgement findings as JSON, each with an exact quote |
| `ToolExecutor` | UserProxyAgent | Runs the tools the Analyst calls (`get_document`, `get_rule_findings`) |
| `Reviewer` | AssistantAgent | Checks the draft against the document: removes duplicates of rule findings, unsupported claims and weak findings, then ends the review |

The tools give the Analyst the **whole** document, so nothing is cut off. The rule results tell it which issues are already covered, so it doesn't repeat them.

In [ ]:
import autogen
from autogen import AssistantAgent, GroupChat, GroupChatManager, UserProxyAgent, register_function

print("AG2 version:", getattr(autogen, "__version__", "unknown"))

FINDING_FORMAT = """{"doc_id": "...", "findings": [
  {"issue_type": "OUTDATED_REFERENCE | WEAK_CONTROL | INCONSISTENCY | OTHER",
   "evidence": "<exact words copied from the document>",
   "recommendation": "<one sentence>"}]}"""


def get_document(doc_id: str) -> str:
    """Return the full text and metadata of a document."""
    doc = DOCUMENTS.get(doc_id)
    if doc is None:
        return f"Unknown document {doc_id}. Known: {list(DOCUMENTS)}"
    return json.dumps({"doc_id": doc_id, "as_of": AS_OF.isoformat(), **doc}, indent=2)


def get_rule_findings(doc_id: str) -> str:
    """Return the issues the rule engine already found in a document (these are final)."""
    if doc_id not in DOCUMENTS:
        return f"Unknown document {doc_id}."
    return json.dumps([{"code": f.code, "evidence": f.evidence} for f in check_rules(doc_id)], indent=2)


analyst = AssistantAgent(
    name="Analyst", llm_config=llm_config,
    system_message=f"""You are a compliance analyst. For the document you are given:
1. Call get_document and get_rule_findings.
2. Find JUDGEMENT issues that the rule findings do not already cover:
   - OUTDATED_REFERENCE: relies on a retired, superseded or old document, form or handbook.
   - WEAK_CONTROL: a security or control requirement that falls short of current good practice.
   - INCONSISTENCY: two statements in the document that contradict each other.
   - OTHER: only for a clear, serious compliance problem not covered above.
3. For each issue, copy the evidence WORD FOR WORD from the document. Report only real problems; an
   empty list is a valid answer for a compliant document. Never repeat a rule finding.
Reply with ONLY this JSON:
{FINDING_FORMAT}""")

reviewer = AssistantAgent(
    name="Reviewer", llm_config=llm_config,
    system_message=f"""You are the compliance reviewer. Check the Analyst's draft against the document text and the rule
findings shown earlier in this conversation. Remove any finding that: repeats a rule finding, has evidence
that is not copied word for word from the document, or is not a real compliance problem. Correct an
issue_type if it is wrong. Do not add speculative findings.
Reply with ONLY the final JSON in the same format, with an extra "removed" list of
{{"evidence": "...", "reason": "..."}}, then the word TERMINATE on its own line:
{FINDING_FORMAT}""")

tool_executor = UserProxyAgent(name="ToolExecutor", human_input_mode="NEVER", code_execution_config=False,
                               default_auto_reply="")
starter = UserProxyAgent(name="Starter", human_input_mode="NEVER", code_execution_config=False,
                         max_consecutive_auto_reply=0)

for fn, desc in [(get_document, "Get a document's full text and metadata by ID."),
                 (get_rule_findings, "Get the final rule-engine findings for a document by ID.")]:
    register_function(fn, caller=analyst, executor=tool_executor, name=fn.__name__, description=desc)

print("Agents ready: Starter, Analyst (2 tools), ToolExecutor, Reviewer")

## Step 7: Review one document

Each document gets its own group chat, so reviews don't mix. `review_document`:

1. runs the group chat with the fixed speaking order;
2. parses the Reviewer's final JSON;
3. **verifies every quote** against the document and rejects any that aren't there;
4. combines the agent findings with the rule findings, then sets severity and routing in code.

Everything is kept for the trace: the full conversation, what the Reviewer removed, and what code rejected.

In [ ]:
import time

TRACES = {}


def review_document(doc_id: str, verbose: bool = False) -> dict:
    chat = GroupChat(agents=[starter, analyst, tool_executor, reviewer], messages=[], max_round=12,
                     speaker_selection_method=make_speaker_selector(starter, analyst, tool_executor, reviewer))
    manager = GroupChatManager(groupchat=chat, llm_config=llm_config,
                               is_termination_msg=lambda m: "TERMINATE" in (m.get("content") or ""))
    start = time.perf_counter()
    starter.initiate_chat(manager, message=f"Review document {doc_id}.", silent=not verbose)
    seconds = round(time.perf_counter() - start, 1)

    final = next((m for m in reversed(chat.messages) if m.get("name") == "Reviewer"), None)
    parsed = extract_json(final.get("content") if final else "") or {}
    agent_findings, rejected = [], []
    for item in parsed.get("findings", []):
        issue_type = str(item.get("issue_type", "OTHER")).upper()
        issue_type = issue_type if issue_type in SEVERITY else "OTHER"
        if verify_evidence(doc_id, item.get("evidence", "")):
            agent_findings.append(Finding(doc_id, issue_type, "agent", item["evidence"], item.get("recommendation", "")))
        else:
            rejected.append(item)                           # quote not in the document: never reported

    findings = check_rules(doc_id) + agent_findings
    severity, routing = route(findings)
    TRACES[doc_id] = {"messages": chat.messages, "reviewer_removed": parsed.get("removed", []),
                      "rejected_by_code": rejected, "parsed": bool(parsed)}
    return {"doc_id": doc_id, "findings": findings, "severity": severity, "routing": routing,
            "seconds": seconds, "rounds": len(chat.messages),
            "tool_calls": sum(len(m.get("tool_calls") or []) for m in chat.messages),
            "removed_by_reviewer": len(parsed.get("removed", [])), "rejected_by_code": len(rejected),
            "parsed": bool(parsed)}


demo = review_document("DOC003", verbose=True)
print(f"\nDOC003 → {demo['severity']}: {demo['routing']}  ({demo['rounds']} messages, {demo['tool_calls']} tool calls, {demo['seconds']} s)")
for f in demo["findings"]:
    print(f"  [{f.source:<5}] {f.severity:<8} {f.code:<32} {f.evidence[:70]}")

## Step 8: Review all documents

In [ ]:
RESULTS = {}
for doc_id in DOCUMENTS:
    RESULTS[doc_id] = review_document(doc_id)
    r = RESULTS[doc_id]
    print(f"{doc_id}: {len(r['findings'])} findings → {r['severity'] or '-':<8} {r['routing']}"
          f"   ({r['rounds']} messages, {r['seconds']} s"
          + (", reply not parsed" if not r["parsed"] else "") + ")")

## Step 9: Baseline: one LLM call, no agents, no rule engine

The same model gets all five full documents, the rules table and `AS_OF`, and is asked to list every issue using the same issue codes. This is what you would get without the system's design. Its quotes are checked the same way.

In [ ]:
baseline_agent = AssistantAgent(name="SinglePass", llm_config=llm_config,
                                system_message="You are a compliance reviewer. Reply with JSON only.")

baseline_prompt = f"""Today is {AS_OF.isoformat()}. Review these documents against the rules and list every compliance issue.

RULES (per document type): {json.dumps(RULES)}
A document is stale if it is older than max_age_months. A section counts as missing if its value is a placeholder.

DOCUMENTS: {json.dumps(DOCUMENTS, indent=1)}

Use exactly these codes: MISSING_SECTION:<section name>, MISSING_SIGNATURE, STALE_DOCUMENT,
OUTDATED_REFERENCE, WEAK_CONTROL, INCONSISTENCY, OTHER.
Reply with ONLY: {{"findings": [{{"doc_id": "...", "code": "...", "evidence": "<exact quote, or what is missing>"}}]}}"""

start = time.perf_counter()
raw = baseline_agent.generate_reply(messages=[{"role": "user", "content": baseline_prompt}])
baseline_seconds = round(time.perf_counter() - start, 1)
baseline = (extract_json(raw if isinstance(raw, str) else (raw or {}).get("content", "")) or {}).get("findings", [])

baseline_pred = {(f.get("doc_id"), f.get("code")) for f in baseline}
baseline_unsupported = [f for f in baseline if f.get("code") in JUDGEMENT_TYPES + ["OTHER"]
                        and not verify_evidence(f.get("doc_id", ""), f.get("evidence", ""))]
print(f"Baseline: {len(baseline)} findings in {baseline_seconds} s; {len(baseline_unsupported)} with a quote not in the document")

## Step 10: Scorecard

Recall against the answer key, split into rule issues and judgement issues, plus the two kinds of error that matter for compliance work: **missed issues**, and **findings that shouldn't be there** (on the compliant DOC004, or with quotes that aren't in the document).

In [ ]:
system_pred = {(f.doc_id, f.code) for r in RESULTS.values() for f in r["findings"]}
rules_pred = {(f.doc_id, f.code) for d in DOCUMENTS for f in check_rules(d)}


def row(name, pred, unsupported):
    rules, judge, total = score(pred, kinds="rule"), score(pred, kinds="judgement"), score(pred)
    on_clean = sum(1 for d, _ in pred if d == "DOC004")
    return [name, f"{rules['found']}/{rules['planted']}", f"{judge['found']}/{judge['planted']}",
            f"{total['found']}/{total['planted']}", len(total["not_in_key"]), on_clean, unsupported]


table = [row("Rules only (code)", rules_pred, 0),
         row("Single LLM call", baseline_pred, len(baseline_unsupported)),
         row("Agents + rules (this system)", system_pred,
             sum(r["rejected_by_code"] for r in RESULTS.values()))]
header = ["Approach", "Rule issues", "Judgement issues", "All planted", "Not in key", "On compliant DOC004", "Unsupported quotes"]
widths = [30, 12, 17, 12, 11, 20, 18]
print("".join(h.ljust(w) for h, w in zip(header, widths)))
for r in table:
    print("".join(str(c).ljust(w) for c, w in zip(r, widths)))

print("\nThis system missed:", score(system_pred)["missed"] or "nothing")
print("Not in the answer key (review these by hand; some may be fair extra findings):", score(system_pred)["not_in_key"] or "none")
print("Single call missed:", score(baseline_pred)["missed"] or "nothing")
print("\nNote: for 'Agents + rules', unsupported quotes were rejected by code and never reported.")

## Step 11: The compliance report

Built from the actual findings (the original demo printed a fixed summary whatever the agents found). Saved as CSV and JSON.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 90)
os.makedirs("outputs", exist_ok=True)

routing = pd.DataFrame([{"doc_id": d, "title": DOCUMENTS[d]["title"], "type": DOCUMENTS[d]["type"],
                         "issues": len(r["findings"]), "top_severity": r["severity"] or "-", "routing": r["routing"]}
                        for d, r in RESULTS.items()])
findings = pd.DataFrame([{**asdict(f), "severity": f.severity} for r in RESULTS.values() for f in r["findings"]])
findings = (findings.assign(rank=findings["severity"].map(SEVERITY_RANK))
                    .sort_values(["doc_id", "rank"], ascending=[True, False]).drop(columns="rank"))

routing.to_csv("outputs/routing.csv", index=False)
findings.to_csv("outputs/findings.csv", index=False)
with open("outputs/report.json", "w") as fh:
    json.dump({"as_of": AS_OF.isoformat(), "routing": routing.to_dict("records"),
               "findings": findings.to_dict("records")}, fh, indent=2)

print(f"Compliance report as of {AS_OF}: {len(findings)} issues across {len(DOCUMENTS)} documents")
print(findings["severity"].value_counts().to_string(), "\n")
routing

In [ ]:
findings[["doc_id", "severity", "code", "source", "evidence", "recommendation"]]

## Step 12: Trace: what each agent did

For every document: the speaking order, which tools the Analyst called, what the Reviewer removed and why, and anything code rejected.

In [ ]:
def show_trace(doc_id: str) -> None:
    t, r = TRACES[doc_id], RESULTS[doc_id]
    order = []
    for m in t["messages"]:
        name = m.get("name", "?")
        calls = [c["function"]["name"] for c in m.get("tool_calls") or []]
        order.append(f"{name}({', '.join(calls)})" if calls else name)
    print(f"===== {doc_id}: {DOCUMENTS[doc_id]['title']}")
    print("  Order:  ", " → ".join(order))
    print(f"  Result:  {len(r['findings'])} findings ({sum(f.source == 'rule' for f in r['findings'])} rule, "
          f"{sum(f.source == 'agent' for f in r['findings'])} agent) → {r['routing']}")
    for item in t["reviewer_removed"]:
        print(f"  Reviewer removed: \"{str(item.get('evidence', ''))[:70]}\" ({item.get('reason', '')})")
    for item in t["rejected_by_code"]:
        print(f"  Code rejected (quote not in document): \"{str(item.get('evidence', ''))[:70]}\"")
    print()


for doc_id in DOCUMENTS:
    show_trace(doc_id)

## Notes

* **Division of labour:** rule checks, severity and routing are code, because they are yes/no decisions that must be consistent and auditable. The agents handle what needs reading: outdated references, weak controls, contradictions.
* **Why a fixed speaking order:** with automatic speaker selection, the original demo let one agent speak five times in a row (four times with an empty reply) and ended only at the round limit. Here each review follows the same path and ends when the Reviewer finishes.
* **Evidence is checked, not trusted:** a finding that can't be quoted from the document is never reported.
* **Answer key caveat:** the documents and planted issues are synthetic, and the answer key is small. Findings "not in the key" aren't automatically wrong; read them in the findings table.
* **Variance:** LLM results vary between runs. Run Steps 8–10 two or three times before quoting numbers.